# 二叉查找树

二叉查找树（Binary Search Tree，简称 BST）是一种非常经典的数据结构，它结合了二叉树和有序查找的特点，在查找、插入和删除操作上具有较高效率。二叉查找树满足以下性质：
对于树中的任意一个节点：

- 左子树上所有节点的值都小于当前节点的值；
- 右子树上所有节点的值都大于当前节点的值；
- 左右子树本身也必须分别是二叉查找树。

根据定义可知，二叉查找树的中序遍历（DFS）序列一定是从小到大排列的。

设二叉查找树的树高为 $h$ ，那么查找、删除、插入三个基本操作的时间复杂度均为 $O(h)$。特别的，如果二叉查找树是平衡的，即 $h\approx\log_2 N$，那么查找、删除、插入的时间复杂度为 $O(\log N)$。

以下我们给出二叉查找树的具体实现。

In [ ]:
class BSTNode:
    def __init__(self, value):
        self.value = value
        self.left = None
        self.right = None

class BinarySearchTree:
    def __init__(self):
        self.root = None

    def insert(self, value):
        self.root = self._insert(self.root, value)

    def _insert(self, node, value):
        if node is None:
            return BSTNode(value)

        if value < node.value:
            node.left = self._insert(node.left, value)
        elif value > node.value:
            node.right = self._insert(node.right, value)

        return node

    def search(self, value):
        return self._search(self.root, value)

    def _search(self, node, value):
        if node is None or node.value == value:
            return node
        if value < node.value:
            return self._search(node.left, value)
        return self._search(node.right, value)

    def delete(self, value):
        self.root = self._delete(self.root, value)

    def _delete(self, node, value):
        if node is None:
            return None

        if value < node.value:
            node.left = self._delete(node.left, value)
        elif value > node.value:
            node.right = self._delete(node.right, value)
        else:
            if node.left is None:
                return node.right
            if node.right is None:
                return node.left

            successor = self._find_min_node(node.right)
            node.value = successor.value
            node.right = self._delete(node.right, successor.value)

        return node

    def _find_min_node(self, node):
        cur = node
        while cur.left is not None:
            cur = cur.left
        return cur

    def inOrderTraversal(self):
        res = []

        def _inOrder(node):
            if node is None:
                return
            _inOrder(node.left)
            res.append(node.value)
            _inOrder(node.right)

        _inOrder(self.root)
        return res


## AVL树
上文提到，如果二叉搜索树是平衡的，即树的高度大约是$\log_2 N$，那么插入、删除、搜索三个基本操作都可在对数复杂度内完成。但是，如果搜索树严重不平衡，那么三个操作的时间复杂度就会严重退化。例如，如果对一个从小到大排列的列表依次进行插入操作建立二叉搜索树，那么得到的树就是完全右偏的树——每个节点要么是叶节点，要么只有右子节点。此时树高为$O(N)$，三种基本操作的时间复杂度也全部退化到$O(N)$。

要防止出现上述性能退化，关键在于二叉搜索树应当是接近于“平衡”的，左右子树的高度不应相差太大。这就引出了AVL树。

我们首先给出平衡因子的定义：
对于一个节点，其平衡因子定义为左子树高度减去右子树高度。如果平衡因子为0，那么称这个节点是平衡的。如果平衡因子大于0，那么节点的左子树比右子树高，称之为“左重”。如果平衡因子小于0，那么节点的右子树比左子树高，称之为“右重”。

AVL 树是一棵满足以下条件的二叉搜索树：
对于任意一个节点，它的平衡因子绝对值不超过1。

### AVL树的旋转与再平衡

AVL树的核心思想是：每次插入或删除后，沿着递归路径从下往上检查节点高度。一旦某个节点的左右子树高度差超过1，就通过局部旋转把这棵子树重新调整平衡。旋转只改变少数几个指针，不会破坏二叉搜索树的大小关系，因此中序遍历结果仍然有序。


#### 右旋

右旋用于处理“左边太高”的情况。设当前失衡节点为 `y`，它的左孩子为 `x`，`x` 的右子树为 `T2`。右旋前的局部结构如下：

```text
        y
       / \
      x   C
     / \
    A   T2
```

右旋时，让 `x` 上升成为这棵子树的新根，让 `y` 下降成为 `x` 的右孩子。原来夹在 `x` 和 `y` 之间的 `T2`，需要接到 `y.left` 上：

```text
        x
       / \
      A   y
         / \
        T2  C
```

对应到代码，就是：

```python
x = y.left
t2 = x.right

x.right = y
y.left = t2
```

为什么这样仍然是二叉搜索树？因为旋转前有 `A < x < T2 < y < C`，旋转后这个相对顺序没有改变，只是根节点从 `y` 变成了 `x`。最后要先更新下降节点 `y` 的高度，再更新上升节点 `x` 的高度，并返回新的子树根 `x`。

#### 左旋

左旋用于处理“右边太高”的情况，过程与右旋对称。设当前失衡节点为 `x`，它的右孩子为 `y`，`y` 的左子树为 `T2`。左旋前：

```text
        x
       / \
      A   y
         / \
        T2  C
```

左旋后：

```text
        y
       / \
      x   C
     / \
    A   T2
```

对应到代码，就是：

```python
y = x.right
t2 = y.left

y.left = x
x.right = t2
```

同理，旋转前后保持 `A < x < T2 < y < C`，所以二叉搜索树性质不会被破坏。最后要先更新下降节点 `x` 的高度，再更新上升节点 `y` 的高度，并返回新的子树根 `y`。

#### 四种失衡类型

AVL树的失衡可以分成四类。判断时先看当前节点 `node` 的平衡因子，再看较高子树根节点的平衡因子。

- LL型：`node` 左重，并且 `node.left` 也左重或平衡。说明新变化发生在左子树的左侧，直接对 `node` 做一次右旋。
- LR型：`node` 左重，但 `node.left` 右重。说明局部结构是“折线”，先对 `node.left` 做左旋，把它转成LL型，再对 `node` 做右旋。
- RR型：`node` 右重，并且 `node.right` 也右重或平衡。说明新变化发生在右子树的右侧，直接对 `node` 做一次左旋。
- RL型：`node` 右重，但 `node.right` 左重。说明局部结构是“折线”，先对 `node.right` 做右旋，把它转成RR型，再对 `node` 做左旋。

也就是说，LL和RR是“直线型”失衡，只需要一次旋转；LR和RL是“折线型”失衡，需要先旋转子节点，再旋转当前节点。

#### rebalance的流程

`_rebalance(node)` 的工作可以概括为三步：

1. 先调用 `_update_height(node)`，因为插入或删除后，当前节点的子树高度可能已经变化。
2. 计算当前节点的平衡因子 `balance = _get_balance(node)`。
3. 根据 `balance` 和子节点的平衡因子判断属于 LL、LR、RR、RL 中的哪一种情况，并返回旋转后的新子树根。

对应到代码逻辑是：

```python
if balance > 1 and self._get_balance(node.left) >= 0:
    return self._right_rotate(node)       # LL

if balance > 1 and self._get_balance(node.left) < 0:
    node.left = self._left_rotate(node.left)
    return self._right_rotate(node)       # LR

if balance < -1 and self._get_balance(node.right) <= 0:
    return self._left_rotate(node)        # RR

if balance < -1 and self._get_balance(node.right) > 0:
    node.right = self._right_rotate(node.right)
    return self._left_rotate(node)        # RL

return node
```

这里必须返回新的子树根，因为旋转后局部根节点可能已经改变。递归调用的上一层会用 `node.left = ...` 或 `node.right = ...` 接住这个返回值；如果旋转发生在整棵树的根节点处，外层的 `insert` 或 `delete` 会把 `self.root` 更新为新的根。

#### AVL代码实现思路

实现AVL树时，可以在普通BST的基础上增加高度维护和再平衡逻辑：

- `AVLNode` 继承 `BSTNode`，额外保存 `height`。空节点高度记为0，叶节点高度记为1。
- `_height(node)` 统一处理空节点，避免每次计算高度前都判断 `None`。
- `_update_height(node)` 用左右子树高度更新当前节点高度：`1 + max(left_height, right_height)`。
- `_get_balance(node)` 返回左子树高度减右子树高度，用来判断是否失衡以及失衡方向。
- `_left_rotate` 和 `_right_rotate` 只负责局部指针调整、更新高度、返回新的子树根。
- `_rebalance` 负责统一判断四种失衡类型，并调用对应的旋转操作。
- `_insert` 仍然先按BST规则递归插入；递归返回时，对路径上的每个祖先节点执行 `_rebalance`。
- `_delete` 也先按BST规则删除节点；如果被删节点有两个孩子，就用右子树中的最小节点作为后继替换当前值，再删除后继节点；递归返回时同样执行 `_rebalance`。

插入和删除都要在递归返回阶段做再平衡，这是因为真正发生高度变化的位置在更深层。只有从下往上更新高度并检查平衡因子，才能保证整条路径上的所有祖先节点重新满足AVL条件。

In [ ]:
class AVLNode(BSTNode):
    def __init__(self, value):
        super().__init__(value)
        self.height = 1


class AVLTree(BinarySearchTree):
    def _height(self, node):
        if node is None:
            return 0
        return node.height

    def _update_height(self, node):
        node.height = 1 + max(
            self._height(node.left),
            self._height(node.right)
        )

    def _get_balance(self, node):
        if node is None:
            return 0
        return self._height(node.left) - self._height(node.right)

    def _right_rotate(self, y):
        x = y.left
        t2 = x.right

        # 执行右旋
        x.right = y
        y.left = t2

        # 先更新 y，再更新 x
        self._update_height(y)
        self._update_height(x)

        return x

    def _left_rotate(self, x):
        y = x.right
        t2 = y.left

        # 执行左旋
        y.left = x
        x.right = t2

        # 先更新 x，再更新 y
        self._update_height(x)
        self._update_height(y)

        return y

    def _rebalance(self, node):
        self._update_height(node)

        balance = self._get_balance(node)

        # LL 型：左子树的左侧过高
        if balance > 1 and self._get_balance(node.left) >= 0:
            return self._right_rotate(node)

        # LR 型：左子树的右侧过高
        if balance > 1 and self._get_balance(node.left) < 0:
            node.left = self._left_rotate(node.left)
            return self._right_rotate(node)

        # RR 型：右子树的右侧过高
        if balance < -1 and self._get_balance(node.right) <= 0:
            return self._left_rotate(node)

        # RL 型：右子树的左侧过高
        if balance < -1 and self._get_balance(node.right) > 0:
            node.right = self._right_rotate(node.right)
            return self._left_rotate(node)

        return node

    def _insert(self, node, value):
        if node is None:
            return AVLNode(value)

        if value < node.value:
            node.left = self._insert(node.left, value)
        elif value > node.value:
            node.right = self._insert(node.right, value)
        else:
            # 不插入重复值
            return node

        return self._rebalance(node)

    def _delete(self, node, value):
        if node is None:
            return None

        if value < node.value:
            node.left = self._delete(node.left, value)

        elif value > node.value:
            node.right = self._delete(node.right, value)

        else:
            # 情况 1：没有左孩子
            if node.left is None:
                return node.right

            # 情况 2：没有右孩子
            if node.right is None:
                return node.left

            # 情况 3：左右孩子都有
            successor = self._find_min_node(node.right)
            node.value = successor.value
            node.right = self._delete(node.right, successor.value)

        return self._rebalance(node)
